In [ ]:
# Import libraries
import numpy as np
import hdbscan
import pandas as pd
from sklearn import manifold, cluster
from matplotlib import pyplot as plt
import umap
import umap.plot
from pathlib import Path

In [ ]:
repo_root = Path(__file__).parent.parent
DIST_FILE = repo_root / 'analysis/mash/filtered_distance_matrix.npy'
SAMPLE_FILE = repo_root / 'analysis/samples_for_clustering.txt'


In [ ]:
# Load distance matrix, convert to float64
distance_matrix = np.load(DIST_FILE)
distance_matrix = distance_matrix.astype(np.float64)
print("Matrix loaded!")

# Print shape to verify
print(distance_matrix.shape)

In [4]:
# Create clusterer object
clusterer = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=1, cluster_selection_epsilon= 0.0005, metric='precomputed')

# Fit the clusterer to the data
clusterer.fit(distance_matrix)

In [5]:
# Print the number of clusters found
print(f"Number of clusters found: {max(clusterer.labels_)}")
# Print number of noise points
print(f"Number of noise points: {np.sum(clusterer.labels_ == -1)}")
# Print number of points in each cluster
print(f"Number of points in each cluster: {np.bincount(clusterer.labels_[clusterer.labels_ >= 0])}")

In [6]:
# For each cluster, find the point in the cluster with the smallest distance to other points in the cluster
cluster_centers = []
for cluster_label in np.unique(clusterer.labels_):
    if cluster_label == -1:  # Skip noise points
        continue
    cluster_points = np.where(clusterer.labels_ == cluster_label)[0]
    distances_to_cluster = distance_matrix[cluster_points][:, cluster_points]
    center_index = cluster_points[np.argmin(np.sum(distances_to_cluster, axis=1))]
    cluster_centers.append(center_index)

print(f"Cluster centers: {cluster_centers}")

In [11]:
# Initialize UMAP object
umap_model = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, metric='precomputed', random_state=42)

# Fit the UMAP model to the distance matrix
u = umap_model.fit_transform(distance_matrix)

In [13]:
# Plot the UMAP embedding
umap.plot.points(umap_model, labels=clusterer.labels_)

In [14]:
# Initialize k-means clustering
kmeans = cluster.KMeans(n_clusters=max(clusterer.labels_)+1, random_state=42)

# Fit the k-means model to the UMAP embedding
kmeans.fit_predict(u)

In [15]:
# Plot the k-means clustering results
umap.plot.points(umap_model, labels=kmeans.labels_)

# Compare to the HDBSCAN clustering results
umap.plot.points(umap_model, labels=clusterer.labels_)


In [16]:
# Get the number of samples in each k-means cluster
kmeans_cluster_counts = pd.Series(kmeans.labels_).value_counts().sort_index()
# Print the number of samples in each k-means cluster
print("Number of samples in each k-means cluster:") 
with pd.option_context('display.max_rows', None):
    print(kmeans_cluster_counts)

In [ ]:
# Find the point in each k-means cluster that is closest to the cluster centroid
kmeans_cluster_centers = []
for cluster_label in np.unique(kmeans.labels_):
    cluster_points = np.where(kmeans.labels_ == cluster_label)[0]
    centroid = kmeans.cluster_centers_[cluster_label]
    closest_index = cluster_points[np.argmin(np.linalg.norm(u[cluster_points] - centroid, axis=1))]
    kmeans_cluster_centers.append(closest_index)


In [ ]:
# Load sample ids corresponding to the samples in the distance matrix
sample_ids = np.loadtxt(SAMPLE_FILE, dtype=str)

# Get the ids of k-means cluster centers
kmeans_cluster_center_ids = sample_ids[kmeans_cluster_centers]

# Save the ids of k-means cluster centers to a text file
np.savetxt(repo_root / 'analysis/mash/clustering/kmeans_cluster_centers.txt', kmeans_cluster_center_ids, fmt='%s')

# Do the same for HDBSCAN cluster centers
hdbscan_cluster_center_ids = sample_ids[cluster_centers]
np.savetxt(repo_root / 'analysis/mash/clustering/hdbscan_cluster_centers.txt', hdbscan_cluster_center_ids, fmt='%s')